# 07 — Taiwan Biobank external validation

The pooled model is applied to a cohort it has never seen. Taiwan Biobank
measures no fasting insulin, so HOMA-IR — and therefore the observed `IR` label —
cannot be computed for it. **There is no ground truth here and no AUC.** What
this stage produces instead is a *predicted* prevalence, a characterisation of
the predicted-positive group, and two distributional comparisons that ask
whether the predictions behave like real labels.

Produces thesis Table `TWBstat` and Figures `box-stats2`, `TGHDL` and `qq`, plus
the 21% predicted prevalence and the Cohen's *d* quoted in the Discussion. It
also writes `TWB_with_IR.parquet`, which **stage 08 consumes**: the methylation
subset is the 1,199 participants carrying a `MET_ID`, split by this predicted
label.

**Which model does the scoring.** The top-20 CatBoost from stage 06, not the
241-feature one. The legacy notebook cell left its `folder` variable bound to the
run created three cells earlier, and the run timestamps identify that as `V36`;
scoring settles it, since the top-20 model predicts 19,596 positives and the
241-feature model predicts 19,977, and 19,596 is the published number
(`docs/audit.md` F32).

**Run order.** This notebook appends a fourth sheet to the `stats.xlsx` that
notebook 02 creates, so 02 must have been run first.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pickle

import numpy as np
import pandas as pd
import polars as pl
from scipy.stats import ks_2samp

from src.data.io import output_path, processed_path, repo_root, write_parquet
from src.logging_utils import configure_logging
from src.models.evaluate import cohens_d, predict_labels
from src.viz.figures import (
    plot_feature_boxplots,
    plot_quantile_comparison,
    plot_tg_hdl_boxplots,
)
from src.viz.tables import descriptive_statistics

configure_logging(ROOT / "logs")

nhanes = pl.read_parquet(processed_path("NHANES_data.parquet"))
knhanes = pl.read_parquet(processed_path("KNHANES_data.parquet"))
twb_clinical = pl.read_parquet(processed_path("TWB_clinical_data.parquet"))
twb_features = pl.read_parquet(processed_path("TWB_race2_features.parquet"))

with open(repo_root() / "models" / "catboost_top20.pkl", "rb") as handle:
    model = pickle.load(handle)

print(f"TWB features {twb_features.shape}, model expects {len(model.feature_names_)} features")

TWB features (92734, 243), model expects 20 features


## Scoring

The label is added as an integer rather than a boolean, matching the dtype the
legacy pipeline wrote, so that the stored table can be compared cell by cell.

In [2]:
labels = predict_labels(model, twb_features)

twb = twb_features.with_columns(IR=pl.Series(labels))
twb = twb.select(sorted(twb.columns))
write_parquet(twb, processed_path("TWB_with_IR.parquet"))

positive = int(labels.sum())
print(f"{twb.height:,} participants scored")
print(f"  IR+ {positive:,} ({positive / twb.height:.1%})")
print(f"  IR- {twb.height - positive:,}")

methylation = twb.filter(pl.col("MET_ID") != "")
methylation_positive = methylation.filter(pl.col("IR") == 1).height
print()
print(f"methylation subset (stage 08 input): {methylation.height:,} participants")
print(f"  IR+ {methylation_positive:,}, IR- {methylation.height - methylation_positive:,}")

2026-09-14 21:30:06 [INFO] src.models.evaluate: Predicted 19596 of 92734 rows positive (21.1%)


92,734 participants scored
  IR+ 19,596 (21.1%)
  IR- 73,138

methylation subset (stage 08 input): 1,199 participants
  IR+ 246, IR- 953


## Cohort characteristics of the predicted groups

The same table as stage 02, over the clinical variables, split by the *predicted*
label. The column order and the `mean±SD` formatting are identical to the other
three sheets; only the meaning of the split differs, which is why the sheet is
named `TWB(predict)`.

Taiwan Biobank has no `FASTING_INSULIN` and no `HOMA-IR`, so those two rows are
absent here.

In [3]:
assert twb["Release_No"].to_list() == twb_clinical["Release_No"].to_list(), (
    "row alignment between the clinical table and the scored table broke"
)
twb_labelled = pl.concat([twb_clinical, twb.select("IR") == 1], how="horizontal")

twb_table = descriptive_statistics(twb_labelled)

with pd.ExcelWriter(
    output_path("stats.xlsx"), mode="a", engine="openpyxl", if_sheet_exists="replace"
) as writer:
    twb_table.to_excel(writer, sheet_name="TWB(predict)", index=False)

twb_table

/opt/anaconda3/envs/bio/lib/python3.12/site-packages/scipy/stats/_distn_infrastructure.py:2071: RuntimeWarning: invalid value encountered in divide
  x = np.asarray((x - loc)/scale, dtype=dtyp)
/opt/anaconda3/envs/bio/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:531: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
2026-09-14 21:30:07 [INFO] src.viz.tables: Descriptive statistics table: 18 rows


,column,All,IR-,IR+,ks_p_value-,ks_p_value+,t_test_p_value,u_test_p_value
0,N,"92,734","73,138","19,596",None,None,None,None
1,Male,"34,859(37.6%)","24,393(33.4%)","10,466(53.4%)",None,None,None,None
2,Female,"57,875(62.4%)","48,745(66.6%)","9,130(46.6%)",None,None,None,None
3,AGE,46.4±13.2,46.2±13.1,47.0±13.7,< 0.01,< 0.01,< 0.01,< 0.01
4,BMI,23.9±3.9,22.7±2.9,28.5±3.8,< 0.01,< 0.01,< 0.01,< 0.01
5,BODY_WAISTLINE,82.1±10.8,79.0±8.5,93.8±10.6,< 0.01,< 0.01,< 0.01,< 0.01
6,BUN,12.8±3.7,12.7±3.6,13.2±3.8,< 0.01,< 0.01,< 0.01,< 0.01
7,CREATININE,0.7±0.3,0.7±0.3,0.8±0.3,< 0.01,< 0.01,< 0.01,< 0.01
8,FASTING_GLUCOSE,93.0±13.1,90.2±6.8,103.4±22.3,< 0.01,< 0.01,< 0.01,< 0.01
9,HBA1C,5.6±0.5,5.5±0.3,6.0±0.8,< 0.01,< 0.01,< 0.01,< 0.01


## Distributions beside the labelled cohorts

The stage 02 boxplot grid with a third cohort added. If the predictions were
noise, the TWB IR+ group would look like the TWB IR− group; instead it separates
on the same variables, in the same direction, as the two measured cohorts.

Two things about this figure are reproduced from the legacy code rather than
corrected (`docs/audit.md` F33):

- The implausible-value filter (`BODY_WAISTLINE < 500`, `T_CHO < 1000`) is
  applied **to Taiwan Biobank only**. Each condition removes exactly one row.
- The statistics table above applies no such filter, so it is computed over all
  92,734 participants while this figure uses 92,732.

The cohort label reads `KNHANES`; the published figure reads `KNHAHES`, the same
typo corrected in stage 02.

In [4]:
BOXPLOT_EXCLUDED = [
    "Release_No", "DIABETES", "SEX", "MET_ID", "FASTING_INSULIN", "HOMA-IR"
]

labelled_panels = pl.concat(
    [
        nhanes.with_columns(RACE=pl.lit("NHANES")),
        knhanes.with_columns(RACE=pl.lit("KNHANES")),
    ]
).select(pl.all().exclude(BOXPLOT_EXCLUDED))
labelled_panels = labelled_panels.select(sorted(labelled_panels.columns))

twb_panel = twb_labelled.select(pl.all().exclude(BOXPLOT_EXCLUDED)).with_columns(
    RACE=pl.lit("TWB\n(IR was predicted)")
)
twb_panel = twb_panel.select(sorted(twb_panel.columns))
twb_panel = twb_panel.filter(pl.col("BODY_WAISTLINE") < 500)
twb_panel = twb_panel.filter(pl.col("T_CHO") < 1000)
twb_panel = twb_panel.with_columns(pl.col(pl.Int64).cast(pl.Float64))

print(f"TWB rows dropped by the implausible-value filter: {twb_labelled.height - twb_panel.height}")

plot_feature_boxplots(
    pl.concat([labelled_panels, twb_panel]),
    "Boxplots of variables by IR and Race (TWB was included)",
    output_path("boxplot_of_variable_by_IR_and_Race(Add TWB).png"),
)

TWB rows dropped by the implausible-value filter: 2


2026-09-14 21:30:10 [INFO] src.viz.figures: Wrote /Users/srwang/Coding/InsulinResistancePredictiveModel/data/output/boxplot_of_variable_by_IR_and_Race(Add TWB).png


PosixPath('/Users/srwang/Coding/InsulinResistancePredictiveModel/data/output/boxplot_of_variable_by_IR_and_Race(Add TWB).png')

## The TG/HDL-C ratio

The triglyceride to HDL-cholesterol ratio is an established surrogate marker of
insulin resistance that the model was never told about — `TG` and `HDL_C` enter
as separate features. If the predicted labels are meaningful, the TWB panel
should separate on this ratio the way the measured cohorts do.

**Correction relative to the thesis figure (D2).** The legacy code built its four
panels from `df1, df1, df1+df2, df3`, so the panel titled `KNHANES` plotted
NHANES. The published figure therefore shows NHANES twice. Here each panel
receives its own cohort, and the plotting function takes a title-to-frame mapping
so the substitution cannot recur. Only panel 2 changes.

In [5]:
plot_tg_hdl_boxplots(
    {
        "NHANES": nhanes,
        "KNHANES": knhanes,
        "NHANES + KNHANES": pl.concat([nhanes, knhanes]),
        "TWB (IR was predicted)": twb,
    },
    output_path("boxplot_of_TG_HDL_C_by_IR_and_Datasets.png"),
)

2026-09-14 21:30:11 [INFO] src.viz.figures: Wrote /Users/srwang/Coding/InsulinResistancePredictiveModel/data/output/boxplot_of_TG_HDL_C_by_IR_and_Datasets.png


PosixPath('/Users/srwang/Coding/InsulinResistancePredictiveModel/data/output/boxplot_of_TG_HDL_C_by_IR_and_Datasets.png')

## How far apart are the two positive groups?

Restricted to insulin-resistant participants, how does the TG/HDL-C ratio of the
*measured* positives compare with that of the *predicted* positives? The Q-Q plot
answers it across the whole distribution; Cohen's *d* reduces it to one number.

The legacy code is inconsistent here: the Q-Q plot uses the log ratio, while the
KS test and Cohen's *d* use the raw ratio. Both are reproduced as written — the
thesis quotes the unlogged *d* — and the asymmetry is flagged rather than
silently harmonised.

In [6]:
RATIO = pl.col("TG") / pl.col("HDL_C")

labelled_positive = pl.concat([nhanes, knhanes]).filter(pl.col("IR"))
twb_positive = twb.filter(pl.col("IR") == 1)


def ratio_values(frame: pl.DataFrame, log: bool) -> np.ndarray:
    """Extract TG/HDL-C for one cohort, optionally on a log scale."""
    expression = np.log(RATIO) if log else RATIO
    return frame.select(expression.alias("value"))["value"].to_numpy()


plot_quantile_comparison(
    ratio_values(labelled_positive, log=True),
    ratio_values(twb_positive, log=True),
    ("NHANES+KNHANES", "TWB"),
    "QQ-Plot of log(TG/HDL-C) : NHANES+KNHANES vs TWB (only IR+)",
    output_path("qqplot_NHANES+KNHANES_vs_TWB.png"),
)

reference_raw = ratio_values(labelled_positive, log=False)
twb_raw = ratio_values(twb_positive, log=False)

ks_result = ks_2samp(reference_raw, twb_raw)
effect_size = cohens_d(reference_raw, twb_raw)

print(f"measured IR+ {len(reference_raw):,}, predicted IR+ {len(twb_raw):,}")
print(f"KS statistic {ks_result.statistic!r}")
print(f"KS p-value   {ks_result.pvalue!r}")
print(f"Cohen's d    {effect_size!r}")

2026-09-14 21:30:12 [INFO] src.viz.figures: Wrote /Users/srwang/Coding/InsulinResistancePredictiveModel/data/output/qqplot_NHANES+KNHANES_vs_TWB.png


measured IR+ 9,409, predicted IR+ 19,596
KS statistic 0.07754907175752629
KS p-value   1.0689005058246168e-33
Cohen's d    -0.1332737447580242


The KS test rejects equality of the two distributions, but on samples this large
it would reject almost any difference. Cohen's *d* of −0.13 is what carries the
interpretation: a negligible-to-small effect, with the predicted-positive group
sitting slightly *higher* on TG/HDL-C than the measured one.

## Gate G7

Every stored artefact of this stage is compared against the legacy project. The
scored table is checked cell by cell; the two statistics are checked at full
float precision rather than at the three decimals the thesis prints; the figures
are compared pixel by pixel. The TG/HDL-C figure is expected to differ, and its
difference is localised and quantified rather than waved through.

In [7]:
from src.validate import compare_frames, compare_tables, reference_frame, reference_output, report

passed = True

passed &= report(
    "TWB_with_IR.parquet",
    compare_frames(twb, reference_frame("TWB_with_IR.parquet"), key="Release_No"),
)

EXPECTED = {
    "scored": 92_734, "IR+": 19_596, "IR-": 73_138,
    "methylation": 1_199, "methylation IR+": 246, "methylation IR-": 953,
}
observed = {
    "scored": twb.height, "IR+": positive, "IR-": twb.height - positive,
    "methylation": methylation.height,
    "methylation IR+": methylation_positive,
    "methylation IR-": methylation.height - methylation_positive,
}
for name, expected in EXPECTED.items():
    ok = observed[name] == expected
    passed &= ok
    print(f"{'PASS' if ok else 'FAIL'}  {name}: {observed[name]:,} (expected {expected:,})")

2026-09-14 21:30:12 [INFO] src.validate: PASS TWB_with_IR.parquet


PASS  TWB_with_IR.parquet
PASS  scored: 92,734 (expected 92,734)
PASS  IR+: 19,596 (expected 19,596)
PASS  IR-: 73,138 (expected 73,138)
PASS  methylation: 1,199 (expected 1,199)
PASS  methylation IR+: 246 (expected 246)
PASS  methylation IR-: 953 (expected 953)


In [8]:
reference_table = pd.read_excel(reference_output("stats.xlsx"), sheet_name="TWB(predict)")
passed &= report(
    "stats.xlsx[TWB(predict)]",
    compare_tables(twb_table, reference_table, ignore_columns=["ks_p_value-"]),
)

moved = [
    (twb_table["column"].iloc[row], reference_table["ks_p_value-"].iloc[row], twb_table["ks_p_value-"].iloc[row])
    for row in range(len(twb_table))
    if str(reference_table["ks_p_value-"].iloc[row]).strip()
    not in ("nan", str(twb_table["ks_p_value-"].iloc[row]).strip())
]
print(f"        ks_p_value- values changed by the D3 correction: {len(moved)}")
for column, before, after in moved:
    print(f"          {column}: {before!r} -> {after!r}")

for sheet in ["NHANES", "KNHANES", "COMBINE"]:
    passed &= report(
        f"stats.xlsx[{sheet}] still intact after the append",
        compare_tables(
            pd.read_excel(output_path("stats.xlsx"), sheet_name=sheet),
            pd.read_excel(reference_output("stats.xlsx"), sheet_name=sheet),
            ignore_columns=["ks_p_value-"],
        ),
    )

2026-09-14 21:30:12 [INFO] src.validate: PASS stats.xlsx[TWB(predict)]


2026-09-14 21:30:12 [INFO] src.validate: PASS stats.xlsx[NHANES] still intact after the append


2026-09-14 21:30:12 [INFO] src.validate: PASS stats.xlsx[KNHANES] still intact after the append


PASS  stats.xlsx[TWB(predict)]
        ks_p_value- values changed by the D3 correction: 0
PASS  stats.xlsx[NHANES] still intact after the append
PASS  stats.xlsx[KNHANES] still intact after the append
PASS  stats.xlsx[COMBINE] still intact after the append


2026-09-14 21:30:12 [INFO] src.validate: PASS stats.xlsx[COMBINE] still intact after the append


In [9]:
STATISTICS = {
    "KS statistic": (ks_result.statistic, 0.07754907175752629),
    "KS p-value": (ks_result.pvalue, 1.0689005058246168e-33),
    "Cohen's d": (effect_size, -0.1332737447580242),
}
for name, (observed_value, expected_value) in STATISTICS.items():
    ok = observed_value == expected_value
    passed &= ok
    print(f"{'PASS' if ok else 'FAIL'}  {name}: {observed_value!r} (expected {expected_value!r})")

PASS  KS statistic: 0.07754907175752629 (expected 0.07754907175752629)
PASS  KS p-value: 1.0689005058246168e-33 (expected 1.0689005058246168e-33)
PASS  Cohen's d: -0.1332737447580242 (expected -0.1332737447580242)


In [10]:
from PIL import Image


def pixel_difference(new_name: str, legacy_name: str) -> np.ndarray | None:
    """Compare two PNGs, returning the boolean mask of differing pixels."""
    new = np.asarray(Image.open(output_path(new_name)).convert("RGB"), dtype=np.int16)
    legacy = np.asarray(Image.open(reference_output(legacy_name)).convert("RGB"), dtype=np.int16)
    if new.shape != legacy.shape:
        print(f"{new_name}: SIZE MISMATCH {new.shape} vs {legacy.shape}")
        return None
    differing = np.abs(new - legacy).sum(axis=2) > 0
    share = 100 * differing.sum() / differing.size
    print(f"{new_name}: {differing.sum():,} of {differing.size:,} pixels differ ({share:.2f}%)")
    return differing


def differing_bands(mask: np.ndarray) -> list[tuple[int, int]]:
    """Collapse differing image rows into contiguous (first, last) bands."""
    rows = np.flatnonzero(mask.any(axis=1))
    if rows.size == 0:
        return []
    breaks = np.flatnonzero(np.diff(rows) > 1)
    starts = np.concatenate([[0], breaks + 1])
    ends = np.concatenate([breaks, [rows.size - 1]])
    return [(int(rows[s]), int(rows[e])) for s, e in zip(starts, ends)]


qq = pixel_difference("qqplot_NHANES+KNHANES_vs_TWB.png", "qqplot_NHANES+KNHANES vs TWB.png")
passed &= qq is not None and qq.sum() == 0

boxes = pixel_difference(
    "boxplot_of_variable_by_IR_and_Race(Add TWB).png",
    "boxplot_of_variable_by_IR_and_Race(Add TWB).png",
)
if boxes is not None:
    print(f"  differing bands (the KNHAHES -> KNHANES label fix): {differing_bands(boxes)}")

qqplot_NHANES+KNHANES_vs_TWB.png: 0 of 480,000 pixels differ (0.00%)


boxplot_of_variable_by_IR_and_Race(Add TWB).png: 690 of 2,250,000 pixels differ (0.03%)
  differing bands (the KNHAHES -> KNHANES label fix): [(379, 388), (731, 740), (1084, 1093), (1436, 1445)]


### The TG/HDL-C figure (D2)

This figure is *expected* to differ, so the pass criterion is not "no pixels
differ" but "the difference is exactly where the correction is". The four panels
are stacked and share an x axis, and panel 3 already contains every KNHANES
observation, so the shared axis limits do not move; a correct fix therefore
disturbs panel 2 and nothing else.

The numbers the corrected panel shows are printed beside the numbers the
published panel showed, so the change is on the record.

In [11]:
tg_hdl = pixel_difference(
    "boxplot_of_TG_HDL_C_by_IR_and_Datasets.png",
    "boxplot_of_TG_HDL_C_by_IR_and_Datasets.png",
)
if tg_hdl is not None:
    height = tg_hdl.shape[0]
    bands = differing_bands(tg_hdl)
    print(f"  differing bands: {bands}")
    for first, last in bands:
        panel = f"{1 + 4 * first // height}-{1 + 4 * last // height}"
        print(f"    rows {first}-{last} lie in panel(s) {panel} of 4")

print()
print("log(TG/HDL-C) quartiles, panel 2:")
for label, frame in [("published (NHANES)", nhanes), ("corrected (KNHANES)", knhanes)]:
    for status in [False, True]:
        values = ratio_values(frame.filter(pl.col("IR") == status), log=True)
        q1, median, q3 = np.percentile(values, [25, 50, 75])
        print(f"  {label:22s} IR{'+' if status else '-'}  n={len(values):>6,}  "
              f"Q1 {q1:.3f}  median {median:.3f}  Q3 {q3:.3f}")

boxplot_of_TG_HDL_C_by_IR_and_Datasets.png: 4,551 of 480,000 pixels differ (0.95%)
  differing bands: [(201, 243), (251, 294)]
    rows 201-243 lie in panel(s) 2-2 of 4
    rows 251-294 lie in panel(s) 2-2 of 4

log(TG/HDL-C) quartiles, panel 2:


  published (NHANES)     IR-  n= 6,477  Q1 0.080  median 0.483  Q3 0.932
  published (NHANES)     IR+  n= 5,183  Q1 0.577  median 1.025  Q3 1.468
  corrected (KNHANES)    IR-  n=10,912  Q1 0.094  median 0.529  Q3 1.000
  corrected (KNHANES)    IR+  n= 4,226  Q1 0.655  median 1.107  Q3 1.551


In [12]:
print()
print("G7 (Taiwan Biobank external validation):", "PASS" if passed else "FAIL")


G7 (Taiwan Biobank external validation): PASS
